# Suite PHY — The physical layout

`Physical` defines the coordinates physical layout gives the logical layout's points and the frames of its diagram, shapes, grids and cells, in diagram units, each an annotation owned through `Logical.Owns` by what it annotates. mbse-basis [CAP-241: physical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-241); its components, which place them, are TypeScript's only.

In [ ]:
import { assert, Bench, text } from "./support.js";
import { JSON as Json, Paths, Validators } from "@mbse/schemas/Framework";
import { Container, Logical, Physical } from "../src/Framework/index.js";

const b = new Bench();
b.store.transaction(Physical.install);
const [L, P] = [b.store.Logical, b.store.Physical];

function at(x: number, y: number): (v: any) => any {
  return (v: any) => v.x(x).y(y);
}

function size(width: number, height: number): (v: any) => any {
  return (v: any) => v.width(width).height(height);
}

/** The owning entry of an annotation of `of`, as `name`. */
function annotation(of: any, name: string): (a: any) => any {
  return (a: any) => a.insert((e: any) => e.name(name).owner(of));
}

function frame(of: any, role: string, x: number, y: number, width: number, height: number, unit: number): any {
  return P.OfFrame((f: any) => f.owner(annotation(of, "frame"))[role]((a: any) => a.insert((e: any) => e[role](of)))
    .absolute(at(x, y)).relative(at(x, y)).size(size(width, height)).unit_length(unit));
}

<a id="PHY-01"></a>
## PHY-01 · The install, after Logical's and Container's, defines the physical layout's schemas, each valid, and run again writes nothing

In [ ]:
const NAMES = ["Physical.OfPoint2", "Physical.OfSize", "Physical.Placement", "Physical.OfPlacement", "Physical.DiagramFrame", "Physical.ShapeFrame", "Physical.GridFrame", "Physical.CellFrame", "Physical.OfFrame"];
const ANNOTATED: Record<string, string> = { "Logical.OfPoint": "placement", "Logical.OfDiagram": "frame",
  "Logical.OfShape": "frame", "Container.OfGrid": "frame", "Container.OfCell": "frame" };
const schemas = NAMES.map((name) => b.store.registered(name));
assert(schemas.every((schema: any) => schema.validate().length === 0));
b.store.transaction(Physical.install);
assert(NAMES.every((name, i) => b.store.registered(name) === schemas[i]));
assert(Object.entries(ANNOTATED).every(([name, member]) => b.store.registered(name).adjacencies.has(member)));
const properties = b.store.registered("Logical.OfDiagram").properties;
assert(properties.has("unit_length") && properties.has("scaling_factor"));

<a id="PHY-02"></a>
## PHY-02 · A point's placement is owned by the point: its absolute and relative coordinates and its direction, and its path the point's

In [ ]:
assert(b.child("", "cpu") === "[{\"kind\":\"grid\",\"container\":\"overview\",\"span\":3,\"grows\":\"rows\",\"cell_rotation\":0},{\"kind\":\"place\",\"shape\":\"cpu\",\"order\":0,\"to\":[0,0]},{\"kind\":\"extents\",\"rows\":1,\"columns\":1}]");
assert(b.layout("cpu", 2n) === "[{\"kind\":\"grid\",\"container\":\"cpu\",\"span\":2,\"grows\":\"rows\",\"cell_rotation\":0}]");
assert(b.child("cpu", "a") === "[{\"kind\":\"place\",\"shape\":\"a\",\"order\":0,\"to\":[0,0]},{\"kind\":\"extents\",\"rows\":1,\"columns\":1}]");
const a = b.shape("cpu.a");
let o: any;
b.store.transaction(() => {
  o = L.OfPoint((p: any) => p.owner(annotation(a, "origin")).origin_of((x: any) => x.insert((e: any) => e.shape(a))));
  P.OfPlacement((q: any) => q.owner(annotation(o, "placement")).point((x: any) => x.insert((e: any) => e.point(o)))
    .absolute(at(4, 3)).relative(at(4, 3)).direction(0));
});
const placement = o.owns({ name: "placement" });
const path = (x: any): string => Json.ToJSON(b.store)(b.store.Paths.OfRoot.as_schema(), Paths.of(x));
assert(text([placement.absolute.x.get(), placement.absolute.y.get(), placement.relative.x.get(), placement.direction.get()]) === "[4,3,4,0]");
assert(text([[...o.placement].map((e: any) => e.placement === placement), [...placement.point].map((e: any) => e.point === o)]) === "[[true],[true]]");
assert(text(path(placement).slice(path(o).length - 13)) === "\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"placement\\\"}, \\\"next\\\": {\\\"target\\\": {}}}}}}}}}}}\"");
assert(text(Json.ToJSON(b.store)(P.OfPlacement.as_schema(), placement)) === "\"{\\\"mutations\\\": [{\\\"upsert\\\": {\\\"exists\\\": [{\\\"symbols\\\": {\\\"0\\\": {\\\"schema\\\": [{\\\"$ref\\\": {\\\"singleton\\\": \\\".\\\", \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"schemas\\\", \\\"o\\\": \\\"\\\", \\\"properties\\\": {\\\"name\\\": \\\"Physical\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"schemas\\\", \\\"o\\\": \\\"\\\", \\\"properties\\\": {\\\"name\\\": \\\"OfPlacement\\\"}, \\\"next\\\": {\\\"target\\\": {}}}}}}}}]}}, \\\"spec\\\": [{\\\"0\\\": [{\\\"as_property\\\": {\\\"name\\\": \\\"absolute\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"value\\\": {\\\"x\\\": 4.0, \\\"y\\\": 3.0}}}]}}, {\\\"as_property\\\": {\\\"name\\\": \\\"relative\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"value\\\": {\\\"x\\\": 4.0, \\\"y\\\": 3.0}}}]}}, {\\\"as_property\\\": {\\\"name\\\": \\\"direction\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"value\\\": 0.0}}]}}, {\\\"as_adjacency\\\": {\\\"name\\\": \\\"owner\\\", \\\"mutations\\\": [{\\\"insert\\\": {\\\"entries\\\": [{\\\"mutations\\\": [{\\\"as_link\\\": {\\\"name\\\": \\\"owner\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"ref\\\": {\\\"path\\\": {\\\"singleton\\\": \\\"Logical.Diagrams\\\", \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"overview\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"cpu.a\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"origin\\\"}, \\\"next\\\": {\\\"target\\\": {}}}}}}}}}}}}]}}, {\\\"as_property\\\": {\\\"name\\\": \\\"name\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"value\\\": \\\"placement\\\"}}]}}]}]}}]}}, {\\\"as_adjacency\\\": {\\\"name\\\": \\\"point\\\", \\\"mutations\\\": [{\\\"insert\\\": {\\\"entries\\\": [{\\\"mutations\\\": [{\\\"as_link\\\": {\\\"name\\\": \\\"point\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"ref\\\": {\\\"path\\\": {\\\"singleton\\\": \\\"Logical.Diagrams\\\", \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"overview\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"cpu.a\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"origin\\\"}, \\\"next\\\": {\\\"target\\\": {}}}}}}}}}}}}]}}]}]}}]}}]}]}]}}]}\"");

<a id="PHY-03"></a>
## PHY-03 · The frames of a diagram, a shape, a grid and a cell, each with its unit length; the diagram's tuning

In [ ]:
const cpu = b.shape("cpu");
const [entry] = [...cpu.grid];
const grid = entry.grid;
const [cell] = [...grid.cells].map((e: any) => e.cell);
b.store.transaction(() => {
  L.OfDiagram(b.overview, (d: any) => d.unit_length(1).scaling_factor(0.5));
  frame(b.overview, "diagram", 0, 0, 6, 4, 1);
  frame(cpu, "shape", 0, 0, 2, 1, 0.5);
  frame(grid, "grid", 0, 0, 2, 1, 0.5);
  frame(cell, "cell", 0, 0, 1, 1, 0.5);
});
const shown = [b.overview, cpu, grid, cell].map((x) => x.owns({ name: "frame" }));
assert(text(shown.map((f: any) => [f.size.width.get(), f.size.height.get(), f.unit_length.get()])) === "[[6,4,1],[2,1,0.5],[2,1,0.5],[1,1,0.5]]");
assert(text([b.overview.unit_length.get(), b.overview.scaling_factor.get(), [...b.overview.frame].map((e: any) => e.frame === shown[0])]) === "[1,0.5,[true]]");
assert(text(Validators.Validate(b.store).Reachable(L.OfDiagram, b.overview)) === "[]");

<a id="PHY-04"></a>
## PHY-04 · One placement per point, as the validator reports

In [ ]:
b.store.transaction(() => P.OfPlacement((q: any) => q.owner(annotation(o, "another"))
  .point((x: any) => x.insert((e: any) => e.point(o))).direction(90)));
assert(text(Validators.Validate(b.store).Reachable(L.OfDiagram, b.overview)) === "[\"unique(placement) violated: entries agreeing on ['point'] differ on ['placement']\"]");

<a id="PHY-05"></a>
## PHY-05 · An annotation goes with what it annotates: a child removed from its container keeps its points' placements, for it is not deleted, and the cell it left takes its frame

In [ ]:
/** How many objects of `schema` the store reaches from its singletons. */
function reached(schema: any): number {
  return b.store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: schema })]))])).length;
}
assert(text([reached(P.OfPlacement), reached(P.OfFrame), [...cpu.owns].map((e: any) => e.name.get())]) === "[2,4,[\"grid\",\"frame\"]]");
assert(b.unchild("cpu", "a") === "[{\"kind\":\"leave\",\"shape\":\"a\",\"from\":[0,0]},{\"kind\":\"extents\",\"rows\":0,\"columns\":0},{\"kind\":\"place\",\"shape\":\"a\",\"order\":1,\"to\":[0,1]},{\"kind\":\"extents\",\"rows\":1,\"columns\":2}]");
assert(text([reached(P.OfPlacement), reached(P.OfFrame), [...cpu.owns].map((e: any) => e.name.get()), [...grid.cells].length]) === "[2,3,[\"grid\",\"frame\"],0]");